In [ ]:
---
layout: post 
title: Platformer Creation Exploration 
description: The bridge from Gamebuilder to code management. 
permalink: /platformer-gravity-sorcerers
hide: true
toc: false
author: Anish Gupta, Samarth Hande, Rohan Sharma
---


## Introduction

Learn how to create platformer games in the OCS game gngine by creating a custom Player class that has gravity and can jump on platforms. This tutorial will cover the following key platformer concepts:

1. Player Movement  
2. Gravity  
3. Collision Detection  
4. Platforms  

In [ ]:
%%js

// GAME_RUNNER: Sorcerers Platformer Game | hide_edit: true,  width: 100%, height: 500px

import GameControl from '/assets/js/GameEnginev1.1/essentials/GameControl.js';
import GameLevelMario from '/assets/js/projects/platformer-gravity-sorcerers/levels/GameLevelMario.js';


export const gameLevelClasses = [GameLevelMario];
export { GameControl };


`PlatformerPlayer` extends the base `Player` class to add gravity, jumping, and platformer-style collision resolution against `Barrier` objects.

---

## Gravity System

Gravity is simulated by maintaining a `verticalVelocity` value that is modified each frame.

### Key Properties

```js
this.verticalVelocity = 0;          // Current vertical speed
this.gravityAcceleration = 0.4;     // How fast the player falls each frame
this.jumpVelocity = 8;              // Upward speed applied on jump
this.isGrounded = false;            // Whether the player is on the ground
```

### Per-Frame Gravity Loop

Each frame, `update()` runs this sequence:

```js
// 1. Subtract gravity from vertical velocity (pulls player down)
this.verticalVelocity -= this.gravityAcceleration;

// 2. Convert to engine velocity (flip sign)
this.velocity.y = -this.verticalVelocity;

// 3. Move, draw, and check collisions
super.move();
super.draw();
super.collisionChecks();

// 4. After collisions, update grounded state
this.isGrounded = this._groundedThisFrame;
```

Gravity keeps subtracting every frame, so the player accelerates downward continuously unless a collision handler sets `_skipGravityThisFrame = true`.

### Jumping

In `updateVelocity()`, a jump is triggered when the up key is pressed **and** the player is grounded (or has barrier support beneath them):

```js
if (upPressed && (this.isGrounded || this._hasBarrierSupport()) && !this._jumpPressedLatch) {
    this.verticalVelocity = this.jumpVelocity;  // Jump upward
    this.isGrounded = false;
    this._jumpPressedLatch = true;               // Prevent hold-to-jump
}
```

`_jumpPressedLatch` prevents the jump from re-triggering while the key is held. It resets when the key is released.

---

## Collision Detection

### Hitbox Insets

Rather than using the full sprite rectangle, `PlatformerPlayer` shrinks the collision area by a configurable percentage on all four sides:

```js
_getPlayerCollisionInsets() {
    // e.g. widthPercentage: 0.1 trims 10% off each side
    return {
        left:   this.width  * percents.width,
        right:  this.width  * percents.width,
        top:    this.height * percents.height,
        bottom: this.height * percents.height,
    };
}
```

The resulting `_getPlayerWorldCollisionBounds()` gives the actual rectangle used for all collision math.

### Detecting a Hit- `isCollision(other)`

This overrides the base `Player` method. It computes two trimmed rectangles- one for the player, one for the `other` object- and checks if they overlap:

```js
const hit = (
    thisRect.left   < otherRect.right  &&
    thisRect.right  > otherRect.left   &&
    thisRect.top    < otherRect.bottom &&
    thisRect.bottom > otherRect.top
);
```

It also computes **directional touch flags** for both sides of the collision:

```js
touchPoints.this.top    // Player's top edge is touching the other's top
touchPoints.this.bottom // Player's bottom edge is touching the other's bottom
touchPoints.this.left   // ...and so on
```

These flags tell `handleCollisionState()` *how* the objects are touching, not just *that* they're touching.

---

## Collision Resolution- `handleCollisionState()`

Resolution behaves differently depending on whether the colliding object is a `Barrier`.

### Non-Barrier Objects

Defers to the base class:

```js
super.handleCollisionState();
```

### Barrier Objects

Four cases are handled:

#### 1. Landing on Top of a Barrier
```js
if (touchPoints.top && this.verticalVelocity <= 0) {
    this.position.y = otherObject.y - (this.height - insets.bottom); // Snap to surface
    this.verticalVelocity = 0;
    this.isGrounded = true;
    this._skipGravityThisFrame = true;  // Don't re-apply gravity this frame
}
```
The guard `verticalVelocity <= 0` is important- it prevents this block from running *on the same frame the player jumps*, which would cancel the jump immediately.

#### 2. Hitting the Underside of a Barrier
```js
if (touchPoints.bottom) {
    this.position.y = otherObject.y + otherObject.height - insets.top; // Push below
    this.verticalVelocity = 0;  // Kill upward momentum
}
```

#### 3. Left Wall
```js
if (touchPoints.left) {
    this.position.x = otherObject.x - (this.width - insets.right);
}
```

#### 4. Right Wall
```js
if (touchPoints.right) {
    this.position.x = otherObject.x + otherObject.width - insets.left;
}
```

---

## Supporting: Barrier Detection While Airborne

`_hasBarrierSupport()` checks if the player is standing on a `Barrier` even before a full collision event fires- useful for edge cases where the player is at rest on a platform but `isGrounded` may not yet reflect that:

```js
const standingOnTop = Math.abs(playerBounds.bottom - barrierTop) <= supportTolerance; // 4px
```

This ensures a jump can still be initiated from a platform surface even across a single frame gap.